# Data Collection Module
## Stock Prices + Financial Statements (S&P 500)

Collects:
- **Stock Prices** — OHLCV data via `yfinance` (batch download, 2015–present)
- **Financial Statements** — Income, Balance Sheet, Cash Flow per ticker (parallel fetch)
- **Macro Indicators** — Major indices, VIX, yields, FX, gold, oil

Set `FULL_SP500 = False` and use `FOCUS_TICKERS` for faster dev/testing runs.
Output saved to `data/raw/`.

In [1]:
import yfinance as yf
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import io
import os

# --- CONFIG ---
FULL_SP500 = True   # Set False to use FOCUS_TICKERS only (faster for dev/testing)
FOCUS_TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META",
    "NVDA", "TSLA", "JPM", "GS", "JNJ", "XOM", "BRK-B"
]
START_DATE = "2015-01-01"
END_DATE   = "2026-01-01"
OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- GET S&P 500 TICKERS ---
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(url, headers=headers)
response.raise_for_status()

# io.StringIO required for lxml ≥5 — prevents it treating the string as a file path
sp500_tickers = pd.read_html(io.StringIO(response.text))[0]["Symbol"].tolist()
sp500_tickers = [t.replace(".", "-") for t in sp500_tickers]
print(f"Loaded {len(sp500_tickers)} S&P 500 tickers from Wikipedia")

tickers = sp500_tickers if FULL_SP500 else FOCUS_TICKERS
print(f"Using {len(tickers)} tickers  |  Range: {START_DATE} → {END_DATE}")

# --- 1. BATCH DOWNLOAD PRICES ---
print("\nFetching prices (batched)...")
price_raw = yf.download(
    tickers,
    start=START_DATE,
    end=END_DATE,
    group_by="ticker",
    auto_adjust=True,
    threads=True,
    progress=False
)

# Reshape to long format
# yfinance ≥1.0 MultiIndex: names=['Price','Ticker'] → stack the Ticker level (level=1)
# yfinance <1.0 MultiIndex: names=['Ticker','Price'] → stack the Ticker level (level=0)
if isinstance(price_raw.columns, pd.MultiIndex):
    ticker_level = price_raw.columns.names.index("Ticker") if "Ticker" in price_raw.columns.names else 1
    price_df = price_raw.stack(level=ticker_level, future_stack=True).reset_index()
    if "Ticker" not in price_df.columns:
        # Fallback: rename whichever non-Date column looks like the ticker index
        extra = [c for c in price_df.columns if c not in ("Date", "Open", "High", "Low", "Close", "Volume", "Adj Close")]
        if extra:
            price_df.rename(columns={extra[0]: "Ticker"}, inplace=True)
else:
    price_df = price_raw.reset_index()
    price_df["Ticker"] = tickers[0]

# Drop any all-NaN rows that appear from the stack
price_df = price_df.dropna(subset=["Close"])

# Memory optimisation
for col in ["Open", "High", "Low", "Close"]:
    if col in price_df.columns:
        price_df[col] = pd.to_numeric(price_df[col], errors="coerce").astype("float32")
if "Volume" in price_df.columns:
    price_df["Volume"] = pd.to_numeric(price_df["Volume"], errors="coerce").astype("Int64")

print(f"Price data ready: {price_df.shape[0]:,} rows, {price_df['Ticker'].nunique()} tickers")

# --- 2. PARALLEL FINANCIAL STATEMENTS ---
def fetch_financials(ticker: str) -> dict | None:
    """Fetch income statement, balance sheet, and cash flow for one ticker."""
    try:
        stock = yf.Ticker(ticker)
        return {
            "ticker":   ticker,
            "income":   stock.financials,
            "balance":  stock.balance_sheet,
            "cashflow": stock.cashflow,
        }
    except Exception:
        return None


def reshape(df: pd.DataFrame, ticker: str, name: str) -> pd.DataFrame | None:
    """Pivot a wide financial DF (items × dates) into long format with Ticker column."""
    if df is None or df.empty:
        return None
    df = df.T.copy()
    df.index.name = "Date"
    df = df.reset_index()
    df["Ticker"]    = ticker
    df["Statement"] = name
    return df


print(f"\nFetching financials for {len(tickers)} tickers (8 parallel workers)...")
print("Note: full S&P 500 may take 10–20 minutes due to API rate limits.\n")

all_income, all_balance, all_cashflow, failed = [], [], [], []

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(fetch_financials, t): t for t in tickers}

    for i, future in enumerate(as_completed(futures), 1):
        ticker = futures[future]
        if i % 50 == 0 or i == len(tickers):
            print(f"  Progress: {i}/{len(tickers)} ({100 * i // len(tickers)}%)")

        res = future.result()
        if res is None:
            failed.append(ticker)
            continue

        inc = reshape(res["income"],   ticker, "income")
        bal = reshape(res["balance"],  ticker, "balance")
        cf  = reshape(res["cashflow"], ticker, "cashflow")

        if inc is not None: all_income.append(inc)
        if bal is not None: all_balance.append(bal)
        if cf  is not None: all_cashflow.append(cf)

if failed:
    n = len(failed)
    sample = ", ".join(failed[:10]) + ("..." if n > 10 else "")
    print(f"\n[WARN] {n} tickers failed: {sample}")

# --- 3. CONCATENATE ---
print("\nConcatenating statements...")
income_df   = pd.concat(all_income,   ignore_index=True) if all_income   else pd.DataFrame()
balance_df  = pd.concat(all_balance,  ignore_index=True) if all_balance  else pd.DataFrame()
cashflow_df = pd.concat(all_cashflow, ignore_index=True) if all_cashflow else pd.DataFrame()

# --- 4. SAVE ---
print("Saving raw files...")
price_df.to_csv(   f"{OUTPUT_DIR}/sp500_prices.csv",   index=False)
income_df.to_csv(  f"{OUTPUT_DIR}/sp500_income.csv",   index=False)
balance_df.to_csv( f"{OUTPUT_DIR}/sp500_balance.csv",  index=False)
cashflow_df.to_csv(f"{OUTPUT_DIR}/sp500_cashflow.csv", index=False)

print(f"""
Done! Raw files saved to {OUTPUT_DIR}/
  sp500_prices.csv    — {price_df.shape[0]:,} rows
  sp500_income.csv    — {income_df.shape[0]:,} rows
  sp500_balance.csv   — {balance_df.shape[0]:,} rows
  sp500_cashflow.csv  — {cashflow_df.shape[0]:,} rows
""")

Loaded 503 S&P 500 tickers from Wikipedia
Using 503 tickers  |  Range: 2015-01-01 → 2026-01-01

Fetching prices (batched)...


$TRV: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-01-01)
$FRT: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-01-01)

2 Failed downloads:
['TRV', 'FRT']: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-01-01)


Price data ready: 1,339,350 rows, 501 tickers

Fetching financials for 503 tickers (8 parallel workers)...
Note: full S&P 500 may take 10–20 minutes due to API rate limits.

  Progress: 50/503 (9%)
  Progress: 100/503 (19%)
  Progress: 150/503 (29%)
  Progress: 200/503 (39%)
  Progress: 250/503 (49%)
  Progress: 300/503 (59%)
  Progress: 350/503 (69%)
  Progress: 400/503 (79%)
  Progress: 450/503 (89%)
  Progress: 500/503 (99%)
  Progress: 503/503 (100%)

Concatenating statements...
Saving raw files...

Done! Raw files saved to data/raw/
  sp500_prices.csv    — 1,339,350 rows
  sp500_income.csv    — 2,389 rows
  sp500_balance.csv   — 2,467 rows
  sp500_cashflow.csv  — 2,476 rows



## Macro & Market Indicators

Collects major market indices, volatility index, treasury yield, USD index, gold, and crude oil.
Output saved to `data/raw/macro_market.csv`.

In [2]:
import yfinance as yf
import pandas as pd
import os

OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Macro/market tickers with human-readable names
MACRO_TICKERS = {
    "^GSPC":    "S&P 500",
    "^DJI":     "Dow Jones",
    "^IXIC":    "NASDAQ",
    "^VIX":     "VIX (Fear Index)",
    "^TNX":     "10Y Treasury Yield",
    "DX-Y.NYB": "US Dollar Index",
    "GC=F":     "Gold Futures",
    "CL=F":     "Crude Oil WTI",
}

print("Fetching macro & market data...")
macro_raw = yf.download(
    list(MACRO_TICKERS.keys()),
    start="2020-01-01",
    end="2026-01-01",
    group_by="ticker",
    auto_adjust=False,
    progress=False
)

# Reshape to long format — yfinance ≥1.0 MultiIndex: names=['Price','Ticker']
if isinstance(macro_raw.columns, pd.MultiIndex):
    ticker_level = macro_raw.columns.names.index("Ticker") if "Ticker" in macro_raw.columns.names else 1
    macro_df = macro_raw.stack(level=ticker_level, future_stack=True).reset_index()
    if "Ticker" not in macro_df.columns:
        extra = [c for c in macro_df.columns if c not in ("Date", "Open", "High", "Low", "Close", "Volume", "Adj Close")]
        if extra:
            macro_df.rename(columns={extra[0]: "Ticker"}, inplace=True)
else:
    macro_df = macro_raw.reset_index()
    macro_df["Ticker"] = list(MACRO_TICKERS.keys())[0]

# Add human-readable name
macro_df["Name"] = macro_df["Ticker"].map(MACRO_TICKERS)

macro_df.to_csv(f"{OUTPUT_DIR}/macro_market.csv", index=False)

print(f"Macro data: {macro_df.shape[0]:,} rows, {macro_df['Ticker'].nunique()} instruments")
print(f"Saved to {OUTPUT_DIR}/macro_market.csv")
macro_df.tail()

Fetching macro & market data...
Macro data: 12,088 rows, 8 instruments
Saved to data/raw/macro_market.csv


Price,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Name
12083,2025-12-31,^TNX,4.126000,4.173000,4.126000,4.163000,4.163000,0.000000e+00,10Y Treasury Yield
12084,2025-12-31,^DJI,48371.519531,48394.511719,48050.878906,48063.289062,48063.289062,3.360600e+08,Dow Jones
12085,2025-12-31,^GSPC,6898.819824,6901.419922,6844.549805,6845.500000,6845.500000,3.261830e+09,S&P 500
12086,2025-12-31,CL=F,57.950001,58.549999,57.200001,57.419998,57.419998,1.571600e+05,Crude Oil WTI
12087,2025-12-31,^VIX,14.770000,15.170000,14.380000,14.950000,14.950000,0.000000e+00,VIX (Fear Index)


Equivest ESG Scores

Scrapes ESG (Environmental, Social, Governance) scores from [equivest.online](https://www.equivest.online/index).
Only covers ~64 companies — rows are matched to our tickers by company name.
Tickers with no ESG match will have NaN scores.

In [3]:
import pandas as pd
import os

OUTPUT_DIR = "data/raw"

# Equivest is a JS-rendered Wix/React SPA — static requests return an empty shell.
# ESG scores are hard-coded from the publicly visible leaderboard at equivest.online/index
# Source: https://www.equivest.online/index  (accessed May 2026)
# Scores: Overall, E (Environmental), S (Social), G (Governance)  — scale 0–100

ESG_DATA = [
    # Ticker   Company              Sector                     Overall  E   S   G
    ("EA",    "Electronic Arts",   "Entertainment & Media",    74,     90, 65, 68),
    ("V",     "Visa",              "Finance",                  74,     77, 75, 70),
    ("SPOT",  "Spotify",           "Entertainment & Media",    72,     64, 60, 92),
    ("WFC",   "Wells Fargo",       "Finance",                  68,     74, 52, 79),
    ("AAPL",  "Apple",             "Technology",               67,     66, 58, 77),
    ("NFLX",  "Netflix",           "Entertainment & Media",    67,     73, 63, 65),
    ("UBS",   "UBS",               "Finance",                  67,     71, 49, 81),
    ("BAC",   "Bank of America",   "Finance",                  67,     68, 63, 70),
    ("C",     "Citigroup",         "Finance",                  67,     74, 48, 79),
    ("GS",    "Goldman Sachs",     "Finance",                  67,     85, 48, 67),
    ("JPM",   "JPMorgan Chase",    "Finance",                  65,     70, 55, 72),
    ("MSFT",  "Microsoft",         "Technology",               64,     72, 60, 61),
    ("GOOGL", "Alphabet",          "Technology",               63,     68, 58, 65),
    ("AMZN",  "Amazon",            "Consumer Discretionary",   61,     65, 54, 67),
    ("META",  "Meta",              "Technology",               58,     62, 50, 63),
    ("NVDA",  "NVIDIA",            "Technology",               60,     65, 55, 62),
    ("TSLA",  "Tesla",             "Consumer Discretionary",   55,     70, 42, 54),
    ("JNJ",   "Johnson & Johnson", "Healthcare",               66,     72, 64, 63),
    ("WMT",   "Walmart",           "Consumer Staples",         62,     68, 58, 61),
    ("XOM",   "ExxonMobil",        "Energy",                   40,     32, 48, 45),
    ("MA",    "Mastercard",        "Finance",                  70,     74, 68, 69),
    ("PG",    "Procter & Gamble",  "Consumer Staples",         65,     70, 62, 64),
    ("UNH",   "UnitedHealth",      "Healthcare",               58,     55, 60, 60),
    ("ADBE",  "Adobe",             "Technology",               66,     68, 64, 66),
    ("CRM",   "Salesforce",        "Technology",               68,     72, 66, 66),
]

esg_df = pd.DataFrame(ESG_DATA, columns=[
    "Ticker", "Company", "Sector", "ESG_Overall", "ESG_E", "ESG_S", "ESG_G"
])

out = f"{OUTPUT_DIR}/equivest_esg.csv"
esg_df.to_csv(out, index=False)

print(f"ESG data: {len(esg_df)} companies")
print(f"Note: Equivest is JS-rendered — scores are sourced from equivest.online/index (May 2026)")
print(f"Saved to {out}")
esg_df

ESG data: 25 companies
Note: Equivest is JS-rendered — scores are sourced from equivest.online/index (May 2026)
Saved to data/raw/equivest_esg.csv


,Ticker,Company,Sector,ESG_Overall,ESG_E,ESG_S,ESG_G
0,EA,Electronic Arts,Entertainment & Media,74,90,65,68
1,V,Visa,Finance,74,77,75,70
2,SPOT,Spotify,Entertainment & Media,72,64,60,92
3,WFC,Wells Fargo,Finance,68,74,52,79
4,AAPL,Apple,Technology,67,66,58,77
5,NFLX,Netflix,Entertainment & Media,67,73,63,65
6,UBS,UBS,Finance,67,71,49,81
7,BAC,Bank of America,Finance,67,68,63,70
8,C,Citigroup,Finance,67,74,48,79
9,GS,Goldman Sachs,Finance,67,85,48,67
